In [10]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2


meta_df = pd.read_json("./dataset/meta_AMAZON_FASHION.json.gz", lines=True, compression="gzip")
meta_df.head()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


,title,brand,feature,rank,date,asin,imageURL,imageURLHighRes,description,price,also_view,also_buy,fit,details,similar_item,tech1
0,Slime Time Fall Fest [With CDROM and Collector...,Group Publishing (CO),[Product Dimensions:\n \n8....,"13,052,976inClothing,Shoesamp;Jewelry(",8.70 inches,0764443682,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,XCC Qi promise new spider snake preparing men'...,NaN,NaN,"11,654,581inClothing,Shoesamp;Jewelry(",5 star,1291691480,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Magical Things I Really Do Do Too!,Christopher Manos,[Package Dimensions:\n \n8....,"19,308,073inClothing,ShoesJewelry(",5 star,1940280001,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,[For the professional or amateur magician. Ro...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"Ashes to Ashes, Oranges to Oranges",Flickerlamp Publishing,[Package Dimensions:\n \n8....,"19,734,184inClothing,ShoesJewelry(",5 star,1940735033,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aether & Empire #1 - 2016 First Printing Comic...,NaN,[Package Dimensions:\n \n10...,"10,558,646inClothing,Shoesamp;Jewelry(",5 star,1940967805,[https://images-na.ssl-images-amazon.com/image...,[https://images-na.ssl-images-amazon.com/image...,NaN,$4.50,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
review_df = pd.read_json("./dataset/AMAZON_FASHION.json.gz", lines=True, compression="gzip")
review_df.head()

,overall,verified,reviewTime,reviewerID,asin,reviewerName,reviewText,summary,unixReviewTime,vote,style,image
0,5,True,"10 20, 2014",A1D4G1SNUZWQOT,7106116521,Tracy,Exactly what I needed.,perfect replacements!!,1413763200,NaN,NaN,NaN
1,2,True,"09 28, 2014",A3DDWDH9PX2YX2,7106116521,Sonja Lau,"I agree with the other review, the opening is ...","I agree with the other review, the opening is ...",1411862400,3.0,NaN,NaN
2,4,False,"08 25, 2014",A2MWC41EW7XL15,7106116521,Kathleen,Love these... I am going to order another pack...,My New 'Friends' !!,1408924800,NaN,NaN,NaN
3,2,True,"08 24, 2014",A2UH2QQ275NV45,7106116521,Jodi Stoner,too tiny an opening,Two Stars,1408838400,NaN,NaN,NaN
4,3,False,"07 27, 2014",A89F3LQADZBS5,7106116521,Alexander D.,Okay,Three Stars,1406419200,NaN,NaN,NaN


In [30]:
products_col = ['asin', 'title', 'description', 'brand']
products_review_col = ['asin', 'reviewText']

meta_df = meta_df[products_col]
meta_df['description'] = meta_df['description'].apply(lambda x: " ".join(x) if isinstance(x, list) else x)
meta_df['combined_info'] = meta_df['title'].fillna("") + " " + meta_df['description'].fillna("") + " " + meta_df['brand'].fillna("")
# review_df = review_df[review_df["verified"] == True].sort_values('unixReviewTime').groupby('reviewerID').agg({
#     'reviewText': lambda x: " ".join([str(i) for i in x]),
#     'asin': lambda x: list(x)
# }).reset_index()[products_review_col]



In [ ]:
import numpy as np

data = np.load('processed_data/text_embeddings.npy', allow_pickle=True)
print(data[:5])

In [31]:
meta_df

,asin,title,description,brand,combined_info
0,0764443682,Slime Time Fall Fest [With CDROM and Collector...,NaN,Group Publishing (CO),Slime Time Fall Fest [With CDROM and Collector...
1,1291691480,XCC Qi promise new spider snake preparing men'...,NaN,NaN,XCC Qi promise new spider snake preparing men'...
2,1940280001,Magical Things I Really Do Do Too!,For the professional or amateur magician. Rou...,Christopher Manos,Magical Things I Really Do Do Too! For the pro...
3,1940735033,"Ashes to Ashes, Oranges to Oranges",NaN,Flickerlamp Publishing,"Ashes to Ashes, Oranges to Oranges Flickerlam..."
4,1940967805,Aether & Empire #1 - 2016 First Printing Comic...,NaN,NaN,Aether & Empire #1 - 2016 First Printing Comic...
...,...,...,...,...,...
186632,B01HJGXL4O,JT Women's Elegant Off Shoulder Chiffon Maxi L...,NaN,JT,JT Women's Elegant Off Shoulder Chiffon Maxi L...
186633,B01HJHF97K,Microcosm Retro Vintage Black Crochet Lace One...,NaN,Microcosm,Microcosm Retro Vintage Black Crochet Lace One...
186634,B01HJGJ9LS,Lookatool Classic Plain Vintage Army Military ...,NaN,Lookatool,Lookatool Classic Plain Vintage Army Military ...
186635,B01HJHTH5U,Edith Windsor Women's Deep V-neck Beaded Sequi...,NaN,Edith Windsor,Edith Windsor Women's Deep V-neck Beaded Sequi...


In [1]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cu126
True


In [36]:
import importlib
import streaming
import encoder

importlib.reload(streaming)
importlib.reload(encoder)
from encoder import get_model
from streaming import stream_dataframe_to_milvus, connect_milvus
model = get_model()
connect_milvus()

stream_dataframe_to_milvus(meta_df, model, "product_meta", batch_size=500)
# stream_dataframe_to_milvus(meta_df.astype(str).agg(' '.join, axis=1).tolist(), model, "product_meta", batch_size=500)

[autoreload of streaming failed: Traceback (most recent call last):
  File "C:\Users\PC\AppData\Roaming\Python\Python313\site-packages\IPython\extensions\autoreload.py", line 322, in check
    elif self.deduper_reloader.maybe_reload_module(m):
         ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^^^
  File "C:\Users\PC\AppData\Roaming\Python\Python313\site-packages\IPython\extensions\deduperreload\deduperreload.py", line 546, in maybe_reload_module
    new_source_code = f.read()
  File "c:\Program Files\Python313\Lib\encodings\cp1252.py", line 23, in decode
    return codecs.charmap_decode(input,self.errors,decoding_table)[0]
           ~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 256: character maps to <undefined>
]
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 935.71it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-Mi

✅ Đã kết nối tới Milvus (alias: recommendationsystem).
Bắt đầu stream 186637 dòng vào Milvus...
--- Đã nạp được 500/186637 dòng ---
--- Đã nạp được 1000/186637 dòng ---


KeyboardInterrupt: 